# Torus Age+Disease Longitudinal Visualization

This notebook is for the new experiment:
- `examples/Torus_subset_100_id_age_progression/longitudinal_age_disease_conditioned`

It combines:
1. Training-data style analysis (age/diagnosis/thickness/bump trends), and
2. Reconstruction + one-shot/two-shot + composed longitudinal prediction visualization from trained checkpoints.

Expected saved artifacts (from training):
- `ModelParameters/latest.pth`
- `LatentCodes/latest.pth`
- `OptimizerParameters/latest.pth`
- `TensorBoard/ReconstructionsTrain/...`
- `TensorBoard/ReconstructionsTest/...`


In [ ]:

import json
import math
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import trimesh
import skimage.measure
import scipy.ndimage as ndi
from scipy.spatial import cKDTree

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import sys

ROOT = Path('/home/jakaria/INR/Deep3DComp')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from deep_sdf import data, utils

plt.style.use('default')
pd.set_option('display.max_columns', 200)

EXP_DIR = ROOT / 'examples' / 'Torus_subset_100_id_age_progression' / 'longitudinal_age_disease_conditioned'
SPECS_PATH = EXP_DIR / 'specs.json'
assert SPECS_PATH.exists(), f'Missing specs: {SPECS_PATH}'
SPECS = json.loads(SPECS_PATH.read_text())

CHECKPOINT = 'latest'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Primary model/data settings from specs.
LATENT_SIZE = int(SPECS['CodeLength'])
FLOW_HIDDEN_DIMS = list(SPECS.get('FlowHiddenDims', [256, 256]))
CLAMP_DIST = float(SPECS.get('ClampingDistance', 0.1))
TIME_KEY = str(SPECS.get('LongitudinalTimeKey', 'age_norm'))
TIME_MODE = str(SPECS.get('LongitudinalTimeNormalization', 'none')).lower()
USE_AGE_CONDITIONING = bool(SPECS.get('UseAgeConditioning', False))
AGE_COND_DIM = int(SPECS.get('AgeConditionDim', 0))

age_keys = SPECS.get('AgeConditionKeys', None)
if age_keys is None:
    age_keys = [str(SPECS.get('AgeConditionKey', 'age_norm'))]
elif isinstance(age_keys, str):
    age_keys = [age_keys]
else:
    age_keys = [str(k) for k in age_keys]
AGE_CONDITION_KEYS = [k for k in age_keys if k]

# Checkpoint-driven anchor fitting settings (same family as training-time test eval).
OBS_OPT_STEPS = int(SPECS.get('EvalTestOptimizationSteps', 1000))
OBS_OPT_SAMPLES = int(SPECS.get('EvalTestAnchorNumSamples', 16384))
OBS_OPT_LR = float(SPECS.get('EvalTestAnchorLR', 5e-3))
OBS_INIT_STD = float(SPECS.get('EvalTestAnchorInitStd', 0.01))
OBS_CODE_REG = float(SPECS.get('EvalTestAnchorCodeRegLambda', 1e-4))
CODE_BOUND = SPECS.get('CodeBound', None)

# Mesh extraction controls.
GRID_RES_VIS = int(max(SPECS.get('EvalGridResolution', 192), 160))
GRID_RES_ANALYSIS = 96
SMOOTH_SIGMA = 0.6
ISO_CANDIDATES = [-0.003, -0.0015, 0.0, 0.0015, 0.003]
ISO_HOLDOUT_SAMPLES = 20000
KEEP_LARGEST_COMPONENT = True

# Visualization style.
ALIGN_FOR_VIS = True
ALIGN_MODE = 'rigid'   # 'rigid' or 'centroid'
ALIGN_SAMPLES = 10000
ALIGN_MAX_ITERS = 20
ALIGN_TRIM_QUANTILE = 0.90
VIEW_AXIS = 'z'
INPLANE_ROT_DEG = 90.0
PRED_LIGHT_GREEN = '#b8e8b8'

# Forecast analysis settings.
OBSERVED_COUNTS_TO_SHOW = [1, 2]
COMPOSED_MAX_DT_VIS = float(SPECS.get('EvalTestMaxRolloutDt', 0.0))
if COMPOSED_MAX_DT_VIS <= 0.0:
    COMPOSED_MAX_DT_VIS = 0.05

ANALYSIS_MAX_SUBJECTS = 24    # reduce for speed; set None to use all test subjects
FUTURE_YEARS = [1.0, 2.0, 4.0]

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)

print('Experiment:', EXP_DIR)
print('Device:', DEVICE)
print('Checkpoint:', CHECKPOINT)
print('Time key/mode:', TIME_KEY, TIME_MODE)
print('Age conditioning:', USE_AGE_CONDITIONING, 'dim=', AGE_COND_DIM, 'keys=', AGE_CONDITION_KEYS)
print('Vis grid:', GRID_RES_VIS, '| Analysis grid:', GRID_RES_ANALYSIS)


In [ ]:

# ---- Labels + splits loading ----

def _resolve_path(path_value):
    p = Path(path_value)
    if p.is_absolute() and p.exists():
        return p
    cands = [
        EXP_DIR / path_value,
        ROOT / path_value,
        Path.cwd() / path_value,
    ]
    for c in cands:
        if c.exists():
            return c
    return p


def load_labels_table(labels_path):
    obj = torch.load(labels_path, map_location='cpu')

    if isinstance(obj, dict) and 'records' in obj and isinstance(obj['records'], list):
        records = [r for r in obj['records'] if isinstance(r, dict)]
        df = pd.DataFrame(records)
        label_map = {}
        for r in records:
            k = None
            mp = r.get('mesh_path', None)
            if mp is not None:
                k = Path(mp).stem
            if not k and 'name' in r:
                k = Path(str(r['name'])).stem
            if k:
                label_map[k] = r
        meta = {
            'format': 'records',
            'raw_obj': obj,
        }
        return df, label_map, meta

    if isinstance(obj, dict):
        rows = []
        label_map = {}
        for k, v in obj.items():
            if isinstance(v, dict):
                row = {'scan_key': str(k)}
                row.update(v)
                rows.append(row)
                label_map[str(k)] = v
        df = pd.DataFrame(rows)
        meta = {
            'format': 'dict_map',
            'raw_obj': obj,
        }
        return df, label_map, meta

    raise RuntimeError(f'Unsupported labels format: {type(obj)} at {labels_path}')


TRAIN_SPLIT_PATH = _resolve_path(SPECS['TrainSplit'])
TEST_SPLIT_PATH = _resolve_path(SPECS['TestSplit'])
LABELS_PATH = _resolve_path(SPECS.get('AgeMetadataFile', '/home/jakaria/torus_creation/torus_age_disease_longitudinal_100ids_5tp_flat/labels.pt'))

assert TRAIN_SPLIT_PATH.exists(), f'Missing train split: {TRAIN_SPLIT_PATH}'
assert TEST_SPLIT_PATH.exists(), f'Missing test split: {TEST_SPLIT_PATH}'
assert LABELS_PATH.exists(), f'Missing labels: {LABELS_PATH}'

TRAIN_SPLIT = json.loads(TRAIN_SPLIT_PATH.read_text())
TEST_SPLIT = json.loads(TEST_SPLIT_PATH.read_text())

LABEL_DF_RAW, LABEL_MAP, LABEL_META = load_labels_table(LABELS_PATH)

if 'scan_key' not in LABEL_DF_RAW.columns:
    if 'mesh_path' in LABEL_DF_RAW.columns:
        LABEL_DF_RAW['scan_key'] = LABEL_DF_RAW['mesh_path'].astype(str).map(lambda x: Path(x).stem)
    else:
        LABEL_DF_RAW['scan_key'] = LABEL_DF_RAW.index.astype(str)

# Standardized numeric fields for plotting.
for c in ['diagnosis', 'time_index', 'tau_years', 'age', 'age_norm', 'thickness_t', 'bump_height']:
    if c in LABEL_DF_RAW.columns:
        LABEL_DF_RAW[c] = pd.to_numeric(LABEL_DF_RAW[c], errors='coerce')

# Subject id fallback.
if 'subject_id' not in LABEL_DF_RAW.columns:
    LABEL_DF_RAW['subject_id'] = LABEL_DF_RAW['scan_key'].map(lambda s: '_'.join(str(s).split('_')[:2]))

LABEL_DF = LABEL_DF_RAW.copy()
LABEL_DF = LABEL_DF.sort_values(['subject_id', 'time_index'], na_position='last').reset_index(drop=True)

# age_norm to age conversion helper (for future-age reporting).
age_slope = None
age_intercept = None
if {'age_norm', 'age'}.issubset(LABEL_DF.columns):
    m = LABEL_DF[['age_norm', 'age']].dropna()
    if len(m) >= 2:
        coef = np.polyfit(m['age_norm'].to_numpy(), m['age'].to_numpy(), 1)
        age_slope = float(coef[0])
        age_intercept = float(coef[1])


def years_to_age_norm(years):
    if age_slope is not None and abs(age_slope) > 1e-8:
        return float(years) / age_slope
    # fallback (common synthetic range age 50..90)
    return float(years) / 40.0

print('Labels:', LABELS_PATH)
print('Label format:', LABEL_META['format'])
print('Records:', len(LABEL_DF))
print('Train scans:', len(TRAIN_SPLIT), '| Test scans:', len(TEST_SPLIT))
print('Age conversion slope/intercept:', age_slope, age_intercept)

display(LABEL_DF.head(8))


In [ ]:

# ---- Training-data style cohort plots ----

req_cols = ['diagnosis', 'age', 'thickness_t', 'bump_height']
missing = [c for c in req_cols if c not in LABEL_DF.columns]
if missing:
    raise RuntimeError(f'Missing required columns in labels for cohort plots: {missing}')

healthy = LABEL_DF[LABEL_DF['diagnosis'] == 0].copy()
diseased = LABEL_DF[LABEL_DF['diagnosis'] == 1].copy()

print('Healthy rows:', len(healthy), '| Diseased rows:', len(diseased))
print('mean thickness healthy:', float(healthy['thickness_t'].mean()))
print('mean thickness diseased:', float(diseased['thickness_t'].mean()))
print('mean bump healthy:', float(healthy['bump_height'].mean()))
print('mean bump diseased:', float(diseased['bump_height'].mean()))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

if 'subject_id' in LABEL_DF.columns:
    sub = LABEL_DF[['subject_id', 'diagnosis']].dropna().drop_duplicates('subject_id')
    cnt = sub['diagnosis'].value_counts().sort_index()
    axes[0].bar(['Healthy (0)', 'Diseased (1)'], [cnt.get(0, 0), cnt.get(1, 0)], color=['tab:blue', 'tab:red'])
else:
    cnt = LABEL_DF['diagnosis'].value_counts().sort_index()
    axes[0].bar(['Healthy (0)', 'Diseased (1)'], [cnt.get(0, 0), cnt.get(1, 0)], color=['tab:blue', 'tab:red'])
axes[0].set_title('Subjects by diagnosis')
axes[0].set_ylabel('Count')

for diag, color, label in [(0, 'tab:blue', 'Healthy'), (1, 'tab:red', 'Diseased')]:
    vals = LABEL_DF.loc[LABEL_DF['diagnosis'] == diag, 'age'].dropna().to_numpy()
    if len(vals) > 0:
        axes[1].hist(vals, bins=14, alpha=0.6, label=label, color=color)
axes[1].set_title('Age distribution')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Rows')
axes[1].legend()

axes[2].scatter(diseased['age'], diseased['bump_height'], s=18, alpha=0.45, color='tab:red', label='Diseased')
axes[2].scatter(healthy['age'], healthy['bump_height'], s=18, alpha=0.25, color='tab:blue', label='Healthy')
axes[2].set_title('Bump vs age')
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Bump height')
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:

# ---- Age-binned thickness/bump progression and thickness gap ----

bin_edges = np.arange(50, 91, 2)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

df_b = LABEL_DF.copy()
df_b['age_bin'] = pd.cut(df_b['age'], bins=bin_edges, include_lowest=True, right=False)

agg = (
    df_b.groupby(['diagnosis', 'age_bin'], observed=False)
        .agg(
            n=('subject_id', 'count'),
            thickness_mean=('thickness_t', 'mean'),
            thickness_std=('thickness_t', 'std'),
            bump_mean=('bump_height', 'mean'),
            bump_std=('bump_height', 'std'),
        )
        .reset_index()
)

bin_map = {cat: center for cat, center in zip(df_b['age_bin'].cat.categories, bin_centers)}
agg['age_center'] = agg['age_bin'].map(bin_map).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for diag, color, label in [(0, 'tab:blue', 'Healthy'), (1, 'tab:red', 'Diseased')]:
    a = agg[agg['diagnosis'] == diag].sort_values('age_center')
    x = a['age_center'].to_numpy()

    y_t = a['thickness_mean'].to_numpy()
    s_t = a['thickness_std'].fillna(0.0).to_numpy()
    axes[0].plot(x, y_t, color=color, linewidth=2.5, marker='o', label=label)
    axes[0].fill_between(x, y_t - 0.5 * s_t, y_t + 0.5 * s_t, color=color, alpha=0.20)

    y_b = a['bump_mean'].to_numpy()
    s_b = a['bump_std'].fillna(0.0).to_numpy()
    axes[1].plot(x, y_b, color=color, linewidth=2.5, marker='o', label=label)
    axes[1].fill_between(x, y_b - 0.5 * s_b, y_b + 0.5 * s_b, color=color, alpha=0.20)

axes[0].set_title('Thickness vs age')
axes[0].set_xlabel('Age (bin center)')
axes[0].set_ylabel('thickness_t')
axes[0].legend()

axes[1].set_title('Bump vs age')
axes[1].set_xlabel('Age (bin center)')
axes[1].set_ylabel('bump_height')
axes[1].legend()

plt.tight_layout()
plt.show()

h = agg[agg['diagnosis'] == 0][['age_center', 'thickness_mean']].rename(columns={'thickness_mean': 'th_h'})
d = agg[agg['diagnosis'] == 1][['age_center', 'thickness_mean']].rename(columns={'thickness_mean': 'th_d'})
joined = h.merge(d, on='age_center', how='inner').sort_values('age_center')
joined['gap_h_minus_d'] = joined['th_h'] - joined['th_d']

display(joined.head(12))

plt.figure(figsize=(7, 4))
plt.plot(joined['age_center'], joined['gap_h_minus_d'], marker='o', color='purple', linewidth=2)
plt.axhline(0.0, color='black', linestyle='--', linewidth=1)
plt.title('Thickness gap by age bin (healthy - diseased)')
plt.xlabel('Age (bin center)')
plt.ylabel('Thickness gap')
plt.tight_layout()
plt.show()


In [ ]:

# ---- Finite-difference progression velocities from labels ----

velocity_rows = []
for sid, g in LABEL_DF.sort_values(['subject_id', 'time_index']).groupby('subject_id'):
    g = g.reset_index(drop=True)
    for i in range(1, len(g)):
        if 'tau_years' in g.columns and pd.notna(g.loc[i, 'tau_years']) and pd.notna(g.loc[i - 1, 'tau_years']):
            dt = float(g.loc[i, 'tau_years'] - g.loc[i - 1, 'tau_years'])
        else:
            dt = 1.0
        if dt <= 0:
            continue
        bump_vel = float((g.loc[i, 'bump_height'] - g.loc[i - 1, 'bump_height']) / dt)
        thin_vel = float((g.loc[i - 1, 'thickness_t'] - g.loc[i, 'thickness_t']) / dt)
        velocity_rows.append({
            'subject_id': sid,
            'diagnosis': int(g.loc[i, 'diagnosis']) if pd.notna(g.loc[i, 'diagnosis']) else np.nan,
            'mid_age': 0.5 * float(g.loc[i, 'age'] + g.loc[i - 1, 'age']),
            'bump_velocity': bump_vel,
            'thinning_velocity': thin_vel,
            'dt_years': dt,
        })

df_vel = pd.DataFrame(velocity_rows)
print('Velocity rows:', len(df_vel))
display(df_vel.head())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for diag, color, label in [(0, 'tab:blue', 'Healthy'), (1, 'tab:red', 'Diseased')]:
    v = df_vel[df_vel['diagnosis'] == diag]
    axes[0].scatter(v['mid_age'], v['thinning_velocity'], s=16, alpha=0.32, color=color, label=label)
    axes[1].scatter(v['mid_age'], v['bump_velocity'], s=16, alpha=0.32, color=color, label=label)

axes[0].set_title('Thinning velocity vs age (labels)')
axes[0].set_xlabel('Mid-interval age')
axes[0].set_ylabel('Delta thickness / year')
axes[0].legend()

axes[1].set_title('Bump velocity vs age (labels)')
axes[1].set_xlabel('Mid-interval age')
axes[1].set_ylabel('Delta bump / year')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:

# ---- First-visit checks (same family as data notebook) ----

if 'time_index' in LABEL_DF.columns:
    df_t0 = LABEL_DF[LABEL_DF['time_index'] == 0].copy()
else:
    # fallback: earliest visit per subject
    df_t0 = LABEL_DF.sort_values(['subject_id', 'age']).groupby('subject_id', as_index=False).first()

print('t0 rows:', len(df_t0))

g = (
    df_t0.groupby('diagnosis')
         .agg(
            n=('subject_id', 'count'),
            age_mean=('age', 'mean'),
            thickness_t_mean=('thickness_t', 'mean'),
            thickness_t_min=('thickness_t', 'min'),
            thickness_t_max=('thickness_t', 'max'),
            bump_height_mean=('bump_height', 'mean'),
         )
         .reset_index()
)
display(g)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for diag, color, label in [(0, 'tab:blue', 'Healthy'), (1, 'tab:red', 'Diseased')]:
    vals = df_t0.loc[df_t0['diagnosis'] == diag, 'thickness_t']
    axes[0].hist(vals, bins=12, alpha=0.65, color=color, label=label)
axes[0].set_title('t0 thickness distribution')
axes[0].set_xlabel('thickness_t')
axes[0].set_ylabel('subjects')
axes[0].legend()

for diag, color, label in [(0, 'tab:blue', 'Healthy'), (1, 'tab:red', 'Diseased')]:
    vals = df_t0.loc[df_t0['diagnosis'] == diag, 'bump_height']
    axes[1].hist(vals, bins=12, alpha=0.65, color=color, label=label)
axes[1].set_title('t0 bump distribution')
axes[1].set_xlabel('bump_height')
axes[1].set_ylabel('subjects')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:

# ---- Model + geometry helpers (age+disease conditioned longitudinal flow) ----

SID_RE = re.compile(r"__sid-(\d+)__")
TP_RE = re.compile(r"__tp-(\d+)__")
TORUS_RE = re.compile(r"^(?:ID|id)_(\d+)_t(\d+)$")


def base_id(name):
    return Path(name).stem


def parse_sid_tp(name_or_base):
    b = base_id(name_or_base)
    sm = SID_RE.search(b)
    tm = TP_RE.search(b)
    if sm is not None and tm is not None:
        return sm.group(1), int(tm.group(1))
    mm = TORUS_RE.match(b)
    if mm is not None:
        return mm.group(1), int(mm.group(2))
    raise RuntimeError(f'Cannot parse sid/tp from: {b}')


class TemporalFlowMLP(torch.nn.Module):
    def __init__(self, latent_size, hidden_dims, age_condition_dim=0):
        super().__init__()
        self.age_condition_dim = max(0, int(age_condition_dim))
        dims = [latent_size + 2 + self.age_condition_dim] + list(hidden_dims) + [latent_size]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(torch.nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers.append(torch.nn.ReLU(inplace=True))
        self.net = torch.nn.Sequential(*layers)

    def _prepare_age_condition(self, z, age_cond):
        if self.age_condition_dim <= 0:
            return None
        if age_cond is None:
            return torch.zeros(z.shape[0], self.age_condition_dim, device=z.device, dtype=z.dtype)

        age = age_cond
        if age.dim() == 1:
            age = age.unsqueeze(1)
        age = age.to(device=z.device, dtype=z.dtype)

        if age.shape[1] == self.age_condition_dim:
            return age
        if age.shape[1] == 1 and self.age_condition_dim > 1:
            return age.repeat(1, self.age_condition_dim)

        raise ValueError(f'Invalid age condition width: got {age.shape[1]}, expected {self.age_condition_dim}')

    def forward(self, z, s, t, age_cond=None):
        if s.dim() == 1:
            s = s.unsqueeze(1)
        if t.dim() == 1:
            t = t.unsqueeze(1)
        parts = [z, s, t]
        if self.age_condition_dim > 0:
            parts.append(self._prepare_age_condition(z, age_cond))
        x = torch.cat(parts, dim=1)
        return self.net(x)


def apply_temporal_flow(flow_net, z, s, t, age_cond=None):
    return z + (t - s) * flow_net(z, s, t, age_cond=age_cond)


def load_decoder_and_flow(exp_dir, specs, checkpoint='latest'):
    arch = __import__('networks.' + specs['NetworkArch'], fromlist=['Decoder'])
    decoder = arch.Decoder(int(specs['CodeLength']), **specs['NetworkSpecs']).to(DEVICE)
    flow = TemporalFlowMLP(
        int(specs['CodeLength']),
        list(specs.get('FlowHiddenDims', [256, 256])),
        age_condition_dim=int(specs.get('AgeConditionDim', 0)) if bool(specs.get('UseAgeConditioning', False)) else 0,
    ).to(DEVICE)

    model_dir = exp_dir / 'ModelParameters'
    ckpt_path = model_dir / f'{checkpoint}.pth'
    resolved_name = str(checkpoint)

    if not ckpt_path.exists():
        if str(checkpoint) == 'latest':
            numeric = []
            if model_dir.exists():
                for pth in model_dir.glob('*.pth'):
                    stem = pth.stem
                    if stem.isdigit():
                        numeric.append((int(stem), pth))
            if numeric:
                numeric.sort(key=lambda x: x[0])
                resolved_name = str(numeric[-1][0])
                ckpt_path = numeric[-1][1]
                print(f'[info] latest.pth missing; fallback to checkpoint {resolved_name}.pth')
            else:
                raise FileNotFoundError(
                    f"Missing {model_dir / 'latest.pth'} and no numeric checkpoints found in {model_dir}"
                )
        else:
            raise FileNotFoundError(f'Missing checkpoint: {ckpt_path}')

    payload = torch.load(ckpt_path, map_location=DEVICE)

    dec_state = {k.replace('module.', ''): v for k, v in payload['model_state_dict'].items()}
    decoder.load_state_dict(dec_state, strict=False)
    flow.load_state_dict(payload['flow_state_dict'], strict=True)

    decoder.eval()
    flow.eval()
    return decoder, flow, int(payload.get('epoch', -1)), resolved_name


def scan_record_from_label_map(scan_key):
    rec = LABEL_MAP.get(scan_key, None)
    if rec is None:
        raise KeyError(f'Missing label metadata for scan key: {scan_key}')
    return rec


def condition_vector_from_record(rec):
    if not USE_AGE_CONDITIONING:
        return None
    vals = []
    for k in AGE_CONDITION_KEYS:
        if k not in rec:
            raise KeyError(f"Condition key '{k}' missing in record")
        vals.append(float(rec[k]))
    c = torch.tensor(vals, dtype=torch.float32)
    if AGE_COND_DIM > 0 and c.numel() != AGE_COND_DIM:
        raise RuntimeError(f'Condition dim mismatch: got {c.numel()}, expected {AGE_COND_DIM}')
    return c


def _subject_condition_fit(records):
    # Linear fit between earliest and latest visit, per condition dim (matches trainer behavior).
    if not USE_AGE_CONDITIONING:
        return None
    recs = sorted(records, key=lambda r: float(r['time']))
    c0 = recs[0]['cond'].clone().float()
    c1 = recs[-1]['cond'].clone().float()
    t0 = float(recs[0]['time'])
    t1 = float(recs[-1]['time'])
    dt = t1 - t0
    if abs(dt) <= 1e-8:
        slope = torch.zeros_like(c0)
    else:
        slope = (c1 - c0) / dt
    intercept = c0 - slope * t0
    return {
        't_min': t0,
        't_max': t1,
        'slope': slope,
        'intercept': intercept,
        'fallback': c0,
    }


def condition_at_time(fit, t, device, dtype):
    if fit is None:
        return None
    tt = float(t)
    tt = max(float(fit['t_min']), min(float(fit['t_max']), tt))
    c = fit['intercept'].to(dtype=dtype) + fit['slope'].to(dtype=dtype) * tt
    return c.to(device=device, dtype=dtype).view(1, -1)


def build_split_meta(split_entries, split_name):
    data_source = Path(SPECS['DataSource'])
    data_mesh = Path(SPECS['DataSourceMesh'])

    subject_to_records = {}
    all_times = []

    for item in split_entries:
        b = base_id(item)
        sid, tp = parse_sid_tp(b)

        sdf_path = data_source / f'{b}.npz'
        gt_path = data_mesh / f'{b}.obj'
        if not sdf_path.exists() or not gt_path.exists():
            continue

        label_rec = scan_record_from_label_map(b)
        t_val = float(label_rec[TIME_KEY])
        rec = {
            'name': b,
            'sid': sid,
            'tp_raw': int(tp),
            'tp_user': int(tp + 1),
            'sdf_path': sdf_path,
            'gt_path': gt_path,
            'time': t_val,
            'time_raw': t_val,
            'label': label_rec,
            'diagnosis': int(label_rec.get('diagnosis', -1)) if label_rec.get('diagnosis', None) is not None else -1,
            'age': float(label_rec.get('age', np.nan)) if label_rec.get('age', None) is not None else np.nan,
            'age_norm': float(label_rec.get('age_norm', np.nan)) if label_rec.get('age_norm', None) is not None else np.nan,
        }
        rec['cond'] = condition_vector_from_record(label_rec)

        subject_to_records.setdefault(sid, []).append(rec)
        all_times.append(float(t_val))

    for sid in list(subject_to_records.keys()):
        subject_to_records[sid] = sorted(subject_to_records[sid], key=lambda r: float(r['time']))

    return {
        'name': split_name,
        'subject_to_records': subject_to_records,
        'global_min': float(np.min(all_times)) if all_times else 0.0,
        'global_max': float(np.max(all_times)) if all_times else 1.0,
    }


def load_sdf_ram(record):
    sdf_data = data.read_sdf_samples_into_ram(str(record['sdf_path']))
    sdf_data[0] = sdf_data[0][torch.randperm(sdf_data[0].shape[0])]
    sdf_data[1] = sdf_data[1][torch.randperm(sdf_data[1].shape[0])]
    return sdf_data


def optimize_anchor_from_observations(
    decoder,
    flow,
    latent_size,
    observations,
    clamp_dist,
    num_iterations,
    num_samples,
    lr,
    init_std,
    code_reg_lambda=0.0,
    code_bound=None,
):
    if len(observations) == 0:
        raise ValueError('observations cannot be empty')

    device = next(decoder.parameters()).device
    decoder_was_training = decoder.training
    flow_was_training = flow.training
    decoder.eval()
    flow.eval()

    dec_params = list(decoder.parameters())
    flow_params = list(flow.parameters())
    dec_req = [p.requires_grad for p in dec_params]
    flow_req = [p.requires_grad for p in flow_params]

    for p in dec_params:
        p.requires_grad_(False)
    for p in flow_params:
        p.requires_grad_(False)

    try:
        anchor = torch.empty(1, latent_size, device=device).normal_(0.0, float(init_std))
        anchor.requires_grad_(True)
        opt = torch.optim.Adam([anchor], lr=float(lr))
        l1 = torch.nn.L1Loss(reduction='mean')
        hist = []

        obs_sorted = sorted(observations, key=lambda x: float(x['time']))
        baseline_time = float(obs_sorted[0]['time'])

        fit = None
        if USE_AGE_CONDITIONING:
            fit = _subject_condition_fit([
                {'time': float(o['time']), 'cond': o['cond'].detach().cpu().view(-1)}
                for o in obs_sorted if o.get('cond', None) is not None
            ])

        for _ in range(int(num_iterations)):
            opt.zero_grad()
            loss = 0.0
            for obs in obs_sorted:
                sdf_batch = data.unpack_sdf_samples_from_ram(obs['samples'], int(num_samples)).to(device)
                xyz = sdf_batch[:, :3]
                sdf_gt = sdf_batch[:, 3].unsqueeze(1).clamp(-float(clamp_dist), float(clamp_dist))

                t_obs = float(obs['time'])
                t = torch.full((xyz.shape[0], 1), t_obs, device=device, dtype=anchor.dtype)
                s = torch.full_like(t, baseline_time)

                age_cond = None
                if USE_AGE_CONDITIONING:
                    if obs.get('cond', None) is not None:
                        c = obs['cond'].to(device=device, dtype=anchor.dtype).view(1, -1)
                        age_cond = c.repeat(xyz.shape[0], 1)
                    elif fit is not None:
                        c = condition_at_time(fit, t_obs, device=device, dtype=anchor.dtype)
                        age_cond = c.repeat(xyz.shape[0], 1)

                z_t = apply_temporal_flow(flow, anchor.expand(xyz.shape[0], -1), s, t, age_cond=age_cond)
                pred = decoder(torch.cat([z_t, xyz], dim=1)).clamp(-float(clamp_dist), float(clamp_dist))
                loss = loss + l1(pred, sdf_gt)

            loss = loss / len(obs_sorted)
            if code_reg_lambda is not None and float(code_reg_lambda) > 0.0:
                loss = loss + float(code_reg_lambda) * torch.mean(anchor.pow(2))

            loss.backward()
            opt.step()

            if code_bound is not None:
                with torch.no_grad():
                    b = float(code_bound)
                    if b > 0.0:
                        n = anchor.norm(dim=1, keepdim=True)
                        anchor.mul_(torch.clamp(b / (n + 1e-12), max=1.0))

            hist.append(float(loss.detach().cpu().item()))

        return anchor.detach(), hist
    finally:
        for p, req in zip(dec_params, dec_req):
            p.requires_grad_(req)
        for p, req in zip(flow_params, flow_req):
            p.requires_grad_(req)
        if decoder_was_training:
            decoder.train()
        if flow_was_training:
            flow.train()


def keep_largest_component(mesh_obj):
    parts = mesh_obj.split(only_watertight=False)
    if len(parts) <= 1:
        return mesh_obj
    parts = [p for p in parts if len(p.vertices) > 0 and len(p.faces) > 0]
    if not parts:
        return mesh_obj
    parts.sort(key=lambda m: len(m.vertices), reverse=True)
    return parts[0]


def decode_sdf_grid(decoder, latent_vec, N=128, max_batch=2**18):
    voxel_origin = [-1.0, -1.0, -1.0]
    voxel_size = 2.0 / (int(N) - 1)
    num_samples = int(N) ** 3

    overall = torch.arange(0, num_samples, dtype=torch.long)
    samples = torch.zeros(num_samples, 4, dtype=torch.float32)
    samples[:, 2] = overall % int(N)
    samples[:, 1] = (overall // int(N)) % int(N)
    samples[:, 0] = ((overall // int(N)) // int(N)) % int(N)

    samples[:, 0] = samples[:, 0] * voxel_size + voxel_origin[2]
    samples[:, 1] = samples[:, 1] * voxel_size + voxel_origin[1]
    samples[:, 2] = samples[:, 2] * voxel_size + voxel_origin[0]

    latent = latent_vec.to(next(decoder.parameters()).device)

    head = 0
    while head < num_samples:
        end = min(head + int(max_batch), num_samples)
        xyz = samples[head:end, :3].to(latent.device)
        with torch.no_grad():
            pred = utils.decode_sdf(decoder, latent, xyz).squeeze(1).detach().cpu()
        samples[head:end, 3] = pred
        head = end

    sdf_vals = samples[:, 3].reshape(int(N), int(N), int(N)).numpy()
    return sdf_vals, voxel_origin, voxel_size


def mesh_from_sdf_grid(sdf_vals, voxel_origin, voxel_size, iso_level=0.0, smooth_sigma=0.0, keep_largest=True):
    vol = sdf_vals.astype(np.float32)
    if smooth_sigma is not None and float(smooth_sigma) > 0:
        vol = ndi.gaussian_filter(vol, sigma=float(smooth_sigma))

    try:
        verts, faces, normals, _ = skimage.measure.marching_cubes(
            vol, level=float(iso_level), spacing=[voxel_size] * 3, method='lewiner'
        )
    except ValueError:
        return None

    mesh_points = np.zeros_like(verts)
    mesh_points[:, 0] = voxel_origin[0] + verts[:, 0]
    mesh_points[:, 1] = voxel_origin[1] + verts[:, 1]
    mesh_points[:, 2] = voxel_origin[2] + verts[:, 2]

    m = trimesh.Trimesh(vertices=mesh_points, faces=faces, vertex_normals=normals, process=False)
    if keep_largest:
        m = keep_largest_component(m)
    return m


def choose_iso_level(decoder, latent, sdf_data, candidates, clamp_dist=0.1, holdout_samples=20000):
    if not candidates:
        return 0.0

    device = next(decoder.parameters()).device
    pts = data.unpack_sdf_samples_from_ram(sdf_data, int(holdout_samples)).to(device)
    xyz = pts[:, :3]
    gt = pts[:, 3].clamp(-float(clamp_dist), float(clamp_dist))

    with torch.no_grad():
        pred = utils.decode_sdf(decoder, latent, xyz).squeeze(1)

    gt_sign = torch.sign(gt)
    valid = gt_sign != 0
    if valid.sum().item() == 0:
        return 0.0

    best_iso = float(candidates[0])
    best_acc = -1.0
    for iso in candidates:
        ps = torch.sign(pred - float(iso))
        acc = (ps[valid] == gt_sign[valid]).float().mean().item()
        if acc > best_acc:
            best_acc = acc
            best_iso = float(iso)
    return best_iso


def latent_from_anchor_time(anchor, baseline_t, target_t, cond_t=None):
    t = torch.full((1, 1), float(target_t), device=anchor.device, dtype=anchor.dtype)
    s = torch.full((1, 1), float(baseline_t), device=anchor.device, dtype=anchor.dtype)
    return apply_temporal_flow(flow_net, anchor, s, t, age_cond=cond_t)


def mesh_from_anchor_time(anchor, baseline_t, t_target, cond_t=None, sdf_data_for_iso=None, grid_res=128):
    z_t = latent_from_anchor_time(anchor, baseline_t, t_target, cond_t=cond_t)

    iso_level = 0.0
    if sdf_data_for_iso is not None:
        iso_level = choose_iso_level(
            decoder,
            z_t,
            sdf_data_for_iso,
            ISO_CANDIDATES,
            clamp_dist=CLAMP_DIST,
            holdout_samples=ISO_HOLDOUT_SAMPLES,
        )

    sdf_vals, voxel_origin, voxel_size = decode_sdf_grid(decoder, z_t, N=int(grid_res), max_batch=int(2 ** 18))

    m = mesh_from_sdf_grid(
        sdf_vals,
        voxel_origin,
        voxel_size,
        iso_level=float(iso_level),
        smooth_sigma=float(SMOOTH_SIGMA),
        keep_largest=bool(KEEP_LARGEST_COMPONENT),
    )
    if m is None and float(iso_level) != 0.0:
        m = mesh_from_sdf_grid(
            sdf_vals, voxel_origin, voxel_size,
            iso_level=0.0, smooth_sigma=float(SMOOTH_SIGMA), keep_largest=bool(KEEP_LARGEST_COMPONENT)
        )
    if m is None:
        m = mesh_from_sdf_grid(
            sdf_vals, voxel_origin, voxel_size,
            iso_level=0.0, smooth_sigma=0.0, keep_largest=bool(KEEP_LARGEST_COMPONENT)
        )
    if m is None:
        raise RuntimeError('Mesh creation failed')

    return m, z_t.detach(), float(iso_level)


def compose_latent_interval(z_start, sid_records, t_start, t_end, max_dt=0.05):
    dt_total = float(t_end) - float(t_start)
    if abs(dt_total) <= 1e-12:
        return z_start

    step_lim = max(1e-6, float(max_dt))
    n_steps = max(1, int(np.ceil(abs(dt_total) / step_lim)))
    step = dt_total / float(n_steps)

    # Subject condition fit for sampled-time conditions.
    fit = _subject_condition_fit([
        {'time': float(r['time']), 'cond': r['cond'].detach().cpu().view(-1)}
        for r in sid_records if r.get('cond', None) is not None
    ])

    z = z_start
    cur_t = float(t_start)
    for _ in range(n_steps):
        nxt_t = cur_t + step
        s = torch.full((z.shape[0], 1), cur_t, device=z.device, dtype=z.dtype)
        t = torch.full((z.shape[0], 1), nxt_t, device=z.device, dtype=z.dtype)
        c_next = condition_at_time(fit, nxt_t, device=z.device, dtype=z.dtype)
        z = apply_temporal_flow(flow_net, z, s, t, age_cond=c_next)
        cur_t = nxt_t
    return z


def _rotation_matrix_for_axis(axis, degrees):
    th = np.deg2rad(float(degrees))
    c = np.cos(th)
    s = np.sin(th)
    a = str(axis).lower()
    if a == 'x':
        return np.array([[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]], dtype=np.float32)
    if a == 'y':
        return np.array([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]], dtype=np.float32)
    if a == 'z':
        return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    return np.eye(3, dtype=np.float32)


def _display_vertices(mesh_obj):
    verts = np.asarray(mesh_obj.vertices, dtype=np.float32)
    deg = float(INPLANE_ROT_DEG)
    if abs(deg) <= 1e-8:
        return verts
    rot = _rotation_matrix_for_axis(VIEW_AXIS, deg)
    return verts @ rot.T


def _set_bounds(ax, mins, maxs):
    center = (mins + maxs) / 2.0
    radius = float((maxs - mins).max() / 2.0)
    if radius <= 0:
        radius = 1.0
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def _view_axis(ax):
    mode = str(VIEW_AXIS).lower()
    if mode == 'x':
        ax.view_init(elev=0, azim=0)
    elif mode == 'y':
        ax.view_init(elev=0, azim=90)
    elif mode == 'z':
        ax.view_init(elev=90, azim=0)
    else:
        ax.view_init(elev=20, azim=35)


def _clone_mesh_with_vertices(mesh_obj, vertices):
    return trimesh.Trimesh(vertices=np.asarray(vertices, dtype=np.float32), faces=np.asarray(mesh_obj.faces), process=False)


def _kabsch_rigid(src_pts, dst_pts):
    src = np.asarray(src_pts, dtype=np.float32)
    dst = np.asarray(dst_pts, dtype=np.float32)
    src_mean = src.mean(axis=0)
    dst_mean = dst.mean(axis=0)
    src0 = src - src_mean
    dst0 = dst - dst_mean
    H = src0.T @ dst0
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = dst_mean - src_mean @ R.T
    return R.astype(np.float32), t.astype(np.float32)


def _icp_rigid(src_pts, ref_pts, max_iters=20, trim_quantile=0.90, tol=1e-6):
    src = np.asarray(src_pts, dtype=np.float32)
    ref = np.asarray(ref_pts, dtype=np.float32)
    if src.shape[0] < 8 or ref.shape[0] < 8:
        return np.eye(3, dtype=np.float32), np.zeros(3, dtype=np.float32)

    tree = cKDTree(ref)
    R_tot = np.eye(3, dtype=np.float32)
    t_tot = np.zeros(3, dtype=np.float32)

    for _ in range(int(max_iters)):
        src_cur = src @ R_tot.T + t_tot
        d, idx = tree.query(src_cur, k=1)
        nn = ref[idx]

        if trim_quantile is not None and float(trim_quantile) < 1.0:
            thr = np.quantile(d, float(trim_quantile))
            keep = d <= thr
            if int(keep.sum()) >= 16:
                a = src_cur[keep]
                b = nn[keep]
            else:
                a = src_cur
                b = nn
        else:
            a = src_cur
            b = nn

        dR, dt = _kabsch_rigid(a, b)
        R_tot = dR @ R_tot
        t_tot = t_tot @ dR.T + dt

        if np.linalg.norm(dt) < tol and np.linalg.norm(dR - np.eye(3)) < 1e-4:
            break

    return R_tot, t_tot


def align_mesh_for_display(pred_mesh, ref_mesh):
    pv = np.asarray(pred_mesh.vertices, dtype=np.float32)
    rv = np.asarray(ref_mesh.vertices, dtype=np.float32)

    if str(ALIGN_MODE).lower() == 'centroid':
        shift = rv.mean(axis=0) - pv.mean(axis=0)
        aligned_v = pv + shift
        return _clone_mesh_with_vertices(pred_mesh, aligned_v)

    n = int(ALIGN_SAMPLES)
    p_s = trimesh.sample.sample_surface(pred_mesh, n)[0].astype(np.float32)
    r_s = trimesh.sample.sample_surface(ref_mesh, n)[0].astype(np.float32)
    R, t = _icp_rigid(p_s, r_s, max_iters=int(ALIGN_MAX_ITERS), trim_quantile=float(ALIGN_TRIM_QUANTILE))
    aligned_v = pv @ R.T + t
    return _clone_mesh_with_vertices(pred_mesh, aligned_v)


def draw_mesh(ax, mesh_obj, title, color, mins=None, maxs=None):
    verts = _display_vertices(mesh_obj)
    faces = np.asarray(mesh_obj.faces)
    tris = verts[faces]
    coll = Poly3DCollection(tris, alpha=0.95)
    coll.set_facecolor(color)
    coll.set_edgecolor((0, 0, 0, 0.03))
    coll.set_linewidth(0.1)
    ax.add_collection3d(coll)

    if mins is None or maxs is None:
        mins = verts.min(axis=0)
        maxs = verts.max(axis=0)
    _set_bounds(ax, mins, maxs)

    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    _view_axis(ax)


def plot_rows_gt_pred(rows, suptitle, pred_color=PRED_LIGHT_GREEN):
    n = len(rows)
    fig = plt.figure(figsize=(12, max(4, 3.8 * n)))
    for i, row in enumerate(rows):
        gt_mesh = row['gt_mesh']
        pr_mesh = row['pred_mesh']
        label = row['label']

        if ALIGN_FOR_VIS:
            pr_mesh = align_mesh_for_display(pr_mesh, gt_mesh)

        all_verts = np.concatenate([_display_vertices(gt_mesh), _display_vertices(pr_mesh)], axis=0)
        mins = all_verts.min(axis=0)
        maxs = all_verts.max(axis=0)

        ax_l = fig.add_subplot(n, 2, 2 * i + 1, projection='3d')
        draw_mesh(ax_l, gt_mesh, f'{label} | GT', color='#8fb9d6', mins=mins, maxs=maxs)

        ax_r = fig.add_subplot(n, 2, 2 * i + 2, projection='3d')
        draw_mesh(ax_r, pr_mesh, f'{label} | Pred', color=pred_color, mins=mins, maxs=maxs)

    plt.suptitle(suptitle, fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_three(mesh_a, title_a, mesh_b, title_b, mesh_c, title_c, suptitle):
    all_verts = np.concatenate([
        _display_vertices(mesh_a),
        _display_vertices(mesh_b),
        _display_vertices(mesh_c),
    ], axis=0)
    mins = all_verts.min(axis=0)
    maxs = all_verts.max(axis=0)

    fig = plt.figure(figsize=(16, 5))
    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    draw_mesh(ax1, mesh_a, title_a, color='#8fb9d6', mins=mins, maxs=maxs)
    ax2 = fig.add_subplot(1, 3, 2, projection='3d')
    draw_mesh(ax2, mesh_b, title_b, color=PRED_LIGHT_GREEN, mins=mins, maxs=maxs)
    ax3 = fig.add_subplot(1, 3, 3, projection='3d')
    draw_mesh(ax3, mesh_c, title_c, color='#f2b07f', mins=mins, maxs=maxs)

    plt.suptitle(suptitle, fontsize=14)
    plt.tight_layout()
    plt.show()


def torus_proxy_metrics(mesh_obj):
    # Approximate thickness and bump from torus minor-radius profile.
    v = np.asarray(mesh_obj.vertices, dtype=np.float64)
    rho = np.sqrt(v[:, 0] ** 2 + v[:, 1] ** 2)
    R = np.median(rho)
    minor = np.sqrt((rho - R) ** 2 + v[:, 2] ** 2)

    thickness_proxy = float(np.median(minor))
    bump_proxy = float(np.percentile(minor, 98) - np.percentile(minor, 50))
    return {
        'thickness_proxy': thickness_proxy,
        'bump_proxy': bump_proxy,
    }


def latent_speed_proxy(z_t, t_value, cond_t=None):
    tt = torch.full((z_t.shape[0], 1), float(t_value), device=z_t.device, dtype=z_t.dtype)
    with torch.no_grad():
        v = flow_net(z_t, tt, tt, age_cond=cond_t)
        sp = torch.norm(v, dim=1).mean().item()
    return float(sp)


# Load trained model.
decoder, flow_net, MODEL_EPOCH, CHECKPOINT_RESOLVED = load_decoder_and_flow(EXP_DIR, SPECS, checkpoint=CHECKPOINT)
print('Loaded checkpoint epoch:', MODEL_EPOCH)
print('Checkpoint used:', CHECKPOINT_RESOLVED)


In [ ]:

# ---- Build train/test metadata from split + labels ----

TRAIN_META = build_split_meta(TRAIN_SPLIT, split_name='train')
TEST_META = build_split_meta(TEST_SPLIT, split_name='test')

print('Train subjects:', len(TRAIN_META['subject_to_records']))
print('Test subjects:', len(TEST_META['subject_to_records']))


def pick_subject(meta, diagnosis=None, min_scans=5, explicit_sid=None):
    subject_to_records = meta['subject_to_records']

    if explicit_sid is not None:
        sid_text = str(explicit_sid)
        digits = re.findall(r'\d+', sid_text)
        if not digits:
            raise RuntimeError(f'Cannot parse numeric sid from: {explicit_sid}')
        sid_int = int(digits[0])
        matches = [sid for sid in subject_to_records.keys() if int(sid) == sid_int]
        if not matches:
            raise RuntimeError(f'Requested sid={sid_int} missing in split {meta["name"]}')
        return sorted(matches)[0]

    cands = []
    for sid, recs in subject_to_records.items():
        if len(recs) < int(min_scans):
            continue
        diag = recs[0].get('diagnosis', None)
        if diagnosis is not None and diag != int(diagnosis):
            continue
        cands.append(sid)
    if not cands:
        raise RuntimeError(f'No subject found for split={meta["name"]}, diagnosis={diagnosis}, min_scans={min_scans}')
    return sorted(cands)[0]


def observations_from_records(records, observed_count):
    recs = sorted(records, key=lambda r: float(r['time']))
    obs = []
    for r in recs[:int(observed_count)]:
        obs.append({
            'samples': load_sdf_ram(r),
            'time': float(r['time']),
            'cond': None if r['cond'] is None else r['cond'].clone(),
        })
    return obs


def nearest_record_for_time(records, t_target):
    return min(records, key=lambda r: abs(float(r['time']) - float(t_target)))


def forecast_subject(meta, sid, observed_count=1, composed=False, max_dt=0.05, grid_res=160):
    recs = sorted(meta['subject_to_records'][sid], key=lambda r: float(r['time']))
    obs = observations_from_records(recs, observed_count=observed_count)

    anchor, loss_hist = optimize_anchor_from_observations(
        decoder,
        flow_net,
        LATENT_SIZE,
        obs,
        clamp_dist=CLAMP_DIST,
        num_iterations=OBS_OPT_STEPS,
        num_samples=OBS_OPT_SAMPLES,
        lr=OBS_OPT_LR,
        init_std=OBS_INIT_STD,
        code_reg_lambda=OBS_CODE_REG,
        code_bound=CODE_BOUND,
    )

    baseline_t = float(min(r['time'] for r in recs))

    rows = []
    metrics_rows = []

    z_prev = anchor
    t_prev = baseline_t
    fit = _subject_condition_fit([
        {'time': float(r['time']), 'cond': r['cond'].detach().cpu().view(-1)}
        for r in recs if r.get('cond', None) is not None
    ])

    for idx, r in enumerate(recs):
        t_cur = float(r['time'])
        cond_cur = condition_at_time(fit, t_cur, device=anchor.device, dtype=anchor.dtype)

        if composed and idx > 0:
            z_cur = compose_latent_interval(z_prev, recs, t_prev, t_cur, max_dt=max_dt)
            sdf_ref = load_sdf_ram(r)
            pred_mesh, iso = None, None
            z_for_mesh = z_cur
            iso_level = choose_iso_level(decoder, z_for_mesh, sdf_ref, ISO_CANDIDATES, clamp_dist=CLAMP_DIST, holdout_samples=ISO_HOLDOUT_SAMPLES)
            sdf_vals, voxel_origin, voxel_size = decode_sdf_grid(decoder, z_for_mesh, N=int(grid_res), max_batch=int(2 ** 18))
            pred_mesh = mesh_from_sdf_grid(sdf_vals, voxel_origin, voxel_size, iso_level=float(iso_level), smooth_sigma=float(SMOOTH_SIGMA), keep_largest=bool(KEEP_LARGEST_COMPONENT))
            if pred_mesh is None:
                pred_mesh = mesh_from_sdf_grid(sdf_vals, voxel_origin, voxel_size, iso_level=0.0, smooth_sigma=0.0, keep_largest=bool(KEEP_LARGEST_COMPONENT))
            z_prev = z_cur.detach()
            t_prev = t_cur
            z_t = z_cur.detach()
        else:
            sdf_ref = load_sdf_ram(r)
            pred_mesh, z_t, _ = mesh_from_anchor_time(
                anchor,
                baseline_t,
                t_cur,
                cond_t=cond_cur,
                sdf_data_for_iso=sdf_ref,
                grid_res=grid_res,
            )
            z_prev = z_t.detach()
            t_prev = t_cur

        gt_mesh = trimesh.load(r['gt_path'], process=False)
        if isinstance(gt_mesh, trimesh.Scene):
            gt_mesh = trimesh.util.concatenate(tuple(g for g in gt_mesh.geometry.values()))

        rows.append({
            'gt_mesh': gt_mesh,
            'pred_mesh': pred_mesh,
            'label': f"SID={sid} T{int(r['tp_user'])}",
        })

        pmet = torus_proxy_metrics(pred_mesh)
        gmet = {
            'thickness_gt': float(r['label'].get('thickness_t', np.nan)) if r['label'].get('thickness_t', None) is not None else np.nan,
            'bump_gt': float(r['label'].get('bump_height', np.nan)) if r['label'].get('bump_height', None) is not None else np.nan,
        }
        speed = latent_speed_proxy(z_t, t_cur, cond_t=cond_cur)

        metrics_rows.append({
            'split': meta['name'],
            'subject_id': sid,
            'diagnosis': int(r.get('diagnosis', -1)),
            'tp_user': int(r['tp_user']),
            'time_norm': float(t_cur),
            'age': float(r.get('age', np.nan)),
            'age_norm': float(r.get('age_norm', np.nan)),
            'phase': 'observed' if idx < int(observed_count) else 'forecast',
            'thickness_pred_proxy': pmet['thickness_proxy'],
            'bump_pred_proxy': pmet['bump_proxy'],
            'thickness_gt': gmet['thickness_gt'],
            'bump_gt': gmet['bump_gt'],
            'latent_speed': speed,
        })

    return {
        'anchor': anchor,
        'loss_hist': loss_hist,
        'rows': rows,
        'records': recs,
        'baseline_t': baseline_t,
        'metrics_df': pd.DataFrame(metrics_rows),
    }


# Choose one healthy + one diseased test subject with full scans.
TEST_SID_HEALTHY = pick_subject(TEST_META, diagnosis=0, min_scans=5)
TEST_SID_DISEASED = pick_subject(TEST_META, diagnosis=1, min_scans=5)

print('Selected healthy test SID:', TEST_SID_HEALTHY)
print('Selected diseased test SID:', TEST_SID_DISEASED)


In [ ]:

# ---- GT progression 3D (plotly), healthy vs diseased examples ----

def plot_subject_progression_plotly(meta, sid, title_prefix):
    recs = sorted(meta['subject_to_records'][sid], key=lambda r: float(r['time']))
    n = len(recs)
    fig = make_subplots(
        rows=1,
        cols=n,
        specs=[[{'type': 'scene'} for _ in range(n)]],
        subplot_titles=[
            f"T{int(r['tp_user'])}<br>age={float(r.get('age', np.nan)):.1f}<br>diag={int(r.get('diagnosis', -1))}"
            for r in recs
        ],
        horizontal_spacing=0.01,
    )

    for i, r in enumerate(recs):
        mesh = trimesh.load(r['gt_path'], process=False)
        if isinstance(mesh, trimesh.Scene):
            mesh = trimesh.util.concatenate(tuple(g for g in mesh.geometry.values()))
        v = np.asarray(mesh.vertices)
        f = np.asarray(mesh.faces)

        fig.add_trace(
            go.Mesh3d(
                x=v[:, 0], y=v[:, 1], z=v[:, 2],
                i=f[:, 0], j=f[:, 1], k=f[:, 2],
                color='crimson' if int(r.get('diagnosis', -1)) == 1 else 'steelblue',
                opacity=0.96,
                flatshading=False,
                lighting=dict(ambient=0.55, diffuse=0.60, specular=0.18, roughness=0.72),
                showscale=False,
            ),
            row=1,
            col=i + 1,
        )

        scene_name = 'scene' if i == 0 else f'scene{i + 1}'
        fig.update_layout({
            scene_name: dict(
                aspectmode='data',
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                camera=dict(eye=dict(x=1.25, y=1.25, z=0.95)),
            )
        })

    fig.update_layout(
        title=f'{title_prefix} | GT progression | SID={sid}',
        width=max(1100, 260 * n),
        height=370,
        margin=dict(l=0, r=0, t=45, b=0),
    )
    fig.show()


plot_subject_progression_plotly(TEST_META, TEST_SID_HEALTHY, 'Healthy test subject')
plot_subject_progression_plotly(TEST_META, TEST_SID_DISEASED, 'Diseased test subject')


In [ ]:

# ---- One-shot (observe first visit) GT vs predicted progression ----

one_shot_h = forecast_subject(TEST_META, TEST_SID_HEALTHY, observed_count=1, composed=False, grid_res=GRID_RES_VIS)
one_shot_d = forecast_subject(TEST_META, TEST_SID_DISEASED, observed_count=1, composed=False, grid_res=GRID_RES_VIS)

plot_rows_gt_pred(
    one_shot_h['rows'],
    suptitle=f'One-shot test forecast (observe T1) | Healthy SID={TEST_SID_HEALTHY}'
)
plot_rows_gt_pred(
    one_shot_d['rows'],
    suptitle=f'One-shot test forecast (observe T1) | Diseased SID={TEST_SID_DISEASED}'
)

print('Healthy one-shot anchor final loss:', one_shot_h['loss_hist'][-1] if one_shot_h['loss_hist'] else None)
print('Diseased one-shot anchor final loss:', one_shot_d['loss_hist'][-1] if one_shot_d['loss_hist'] else None)

display(one_shot_h['metrics_df'])
display(one_shot_d['metrics_df'])


In [ ]:

# ---- Two-shot (observe first 2 visits) GT vs predicted progression ----

two_shot_h = forecast_subject(TEST_META, TEST_SID_HEALTHY, observed_count=2, composed=False, grid_res=GRID_RES_VIS)
two_shot_d = forecast_subject(TEST_META, TEST_SID_DISEASED, observed_count=2, composed=False, grid_res=GRID_RES_VIS)

plot_rows_gt_pred(
    two_shot_h['rows'],
    suptitle=f'Two-shot test forecast (observe T1+T2) | Healthy SID={TEST_SID_HEALTHY}'
)
plot_rows_gt_pred(
    two_shot_d['rows'],
    suptitle=f'Two-shot test forecast (observe T1+T2) | Diseased SID={TEST_SID_DISEASED}'
)

print('Healthy two-shot anchor final loss:', two_shot_h['loss_hist'][-1] if two_shot_h['loss_hist'] else None)
print('Diseased two-shot anchor final loss:', two_shot_d['loss_hist'][-1] if two_shot_d['loss_hist'] else None)


In [ ]:

# ---- Composed rollout (step-wise transport) from one-shot anchors ----

one_shot_comp_h = forecast_subject(
    TEST_META,
    TEST_SID_HEALTHY,
    observed_count=1,
    composed=True,
    max_dt=COMPOSED_MAX_DT_VIS,
    grid_res=GRID_RES_VIS,
)
one_shot_comp_d = forecast_subject(
    TEST_META,
    TEST_SID_DISEASED,
    observed_count=1,
    composed=True,
    max_dt=COMPOSED_MAX_DT_VIS,
    grid_res=GRID_RES_VIS,
)

plot_rows_gt_pred(
    one_shot_comp_h['rows'],
    suptitle=f'One-shot composed rollout | Healthy SID={TEST_SID_HEALTHY} | max_dt={COMPOSED_MAX_DT_VIS}'
)
plot_rows_gt_pred(
    one_shot_comp_d['rows'],
    suptitle=f'One-shot composed rollout | Diseased SID={TEST_SID_DISEASED} | max_dt={COMPOSED_MAX_DT_VIS}'
)


In [ ]:

# ---- Interpolation + future-age extrapolation examples ----

def future_targets_for_subject(records, future_years=(1.0, 2.0, 4.0)):
    recs = sorted(records, key=lambda r: float(r['time']))
    t_last = float(recs[-1]['time'])
    out = []
    for y in future_years:
        dt = years_to_age_norm(y)
        out.append({'future_years': float(y), 't_target': t_last + float(dt)})
    return out


def show_interp_and_future(meta, sid, anchor_pack, subject_tag):
    recs = sorted(meta['subject_to_records'][sid], key=lambda r: float(r['time']))
    anchor = anchor_pack['anchor']
    baseline_t = anchor_pack['baseline_t']

    # Interpolation between first two visits.
    t1 = float(recs[0]['time'])
    t2 = float(recs[1]['time'])
    t_mid = 0.5 * (t1 + t2)
    fit = _subject_condition_fit([
        {'time': float(r['time']), 'cond': r['cond'].detach().cpu().view(-1)}
        for r in recs if r.get('cond', None) is not None
    ])
    cond_mid = condition_at_time(fit, t_mid, device=anchor.device, dtype=anchor.dtype)

    nearest_mid = nearest_record_for_time(recs, t_mid)
    pred_mid, _, _ = mesh_from_anchor_time(
        anchor,
        baseline_t,
        t_mid,
        cond_t=cond_mid,
        sdf_data_for_iso=load_sdf_ram(nearest_mid),
        grid_res=GRID_RES_VIS,
    )

    gt1 = trimesh.load(recs[0]['gt_path'], process=False)
    gt2 = trimesh.load(recs[1]['gt_path'], process=False)
    if isinstance(gt1, trimesh.Scene):
        gt1 = trimesh.util.concatenate(tuple(g for g in gt1.geometry.values()))
    if isinstance(gt2, trimesh.Scene):
        gt2 = trimesh.util.concatenate(tuple(g for g in gt2.geometry.values()))

    plot_three(
        gt1,
        f'GT T{int(recs[0]["tp_user"])}',
        pred_mid,
        'Pred mid-age',
        gt2,
        f'GT T{int(recs[1]["tp_user"])}',
        suptitle=f'Interpolation | {subject_tag} | SID={sid}'
    )

    # Future extrapolation beyond last observed scan.
    last_gt = trimesh.load(recs[-1]['gt_path'], process=False)
    prev_gt = trimesh.load(recs[-2]['gt_path'], process=False)
    if isinstance(last_gt, trimesh.Scene):
        last_gt = trimesh.util.concatenate(tuple(g for g in last_gt.geometry.values()))
    if isinstance(prev_gt, trimesh.Scene):
        prev_gt = trimesh.util.concatenate(tuple(g for g in prev_gt.geometry.values()))

    for ft in future_targets_for_subject(recs, FUTURE_YEARS):
        t_tar = float(ft['t_target'])
        cond_tar = condition_at_time(fit, t_tar, device=anchor.device, dtype=anchor.dtype)
        pred_fut, _, _ = mesh_from_anchor_time(
            anchor,
            baseline_t,
            t_tar,
            cond_t=cond_tar,
            sdf_data_for_iso=load_sdf_ram(recs[-1]),
            grid_res=GRID_RES_VIS,
        )

        plot_three(
            prev_gt,
            f'GT T{int(recs[-2]["tp_user"])}',
            last_gt,
            f'GT T{int(recs[-1]["tp_user"])}',
            pred_fut,
            f'Pred +{ft["future_years"]:.1f}y',
            suptitle=f'Future extrapolation | {subject_tag} | SID={sid} | +{ft["future_years"]:.1f} years'
        )


show_interp_and_future(TEST_META, TEST_SID_HEALTHY, one_shot_h, 'Healthy one-shot')
show_interp_and_future(TEST_META, TEST_SID_DISEASED, one_shot_d, 'Diseased one-shot')


In [ ]:

# ---- Quantitative prediction analysis across test subjects ----

all_test_sids = sorted(TEST_META['subject_to_records'].keys())
if ANALYSIS_MAX_SUBJECTS is not None:
    all_test_sids = all_test_sids[:int(ANALYSIS_MAX_SUBJECTS)]

rows_all = []

for sid in all_test_sids:
    try:
        out = forecast_subject(
            TEST_META,
            sid,
            observed_count=1,
            composed=False,
            grid_res=GRID_RES_ANALYSIS,
        )
        rows_all.append(out['metrics_df'])

        # Add future-only forecast points (+years) for phase separation analysis.
        recs = out['records']
        anchor = out['anchor']
        baseline_t = out['baseline_t']

        fit = _subject_condition_fit([
            {'time': float(r['time']), 'cond': r['cond'].detach().cpu().view(-1)}
            for r in recs if r.get('cond', None) is not None
        ])

        for ft in future_targets_for_subject(recs, FUTURE_YEARS):
            t_tar = float(ft['t_target'])
            cond_tar = condition_at_time(fit, t_tar, device=anchor.device, dtype=anchor.dtype)
            pred_mesh, z_t, _ = mesh_from_anchor_time(
                anchor,
                baseline_t,
                t_tar,
                cond_t=cond_tar,
                sdf_data_for_iso=load_sdf_ram(recs[-1]),
                grid_res=GRID_RES_ANALYSIS,
            )
            pm = torus_proxy_metrics(pred_mesh)
            sp = latent_speed_proxy(z_t, t_tar, cond_t=cond_tar)

            rows_all.append(pd.DataFrame([{
                'split': TEST_META['name'],
                'subject_id': sid,
                'diagnosis': int(recs[0].get('diagnosis', -1)),
                'tp_user': np.nan,
                'time_norm': t_tar,
                'age': (age_slope * t_tar + age_intercept) if age_slope is not None else np.nan,
                'age_norm': t_tar,
                'phase': 'future',
                'thickness_pred_proxy': pm['thickness_proxy'],
                'bump_pred_proxy': pm['bump_proxy'],
                'thickness_gt': np.nan,
                'bump_gt': np.nan,
                'latent_speed': sp,
                'future_years': float(ft['future_years']),
            }]))
    except Exception as e:
        print(f'[warn] subject {sid} skipped due to error: {e}')

if len(rows_all) == 0:
    raise RuntimeError('No test prediction rows were generated.')

PRED_DF = pd.concat(rows_all, ignore_index=True)

print('Prediction rows:', len(PRED_DF), '| subjects used:', PRED_DF['subject_id'].nunique())
display(PRED_DF.head(12))


In [ ]:

# ---- Predicted trend plots: thickness/bump/speed vs age, split by diagnosis ----

def age_binned_curve(df, y_col, bin_edges=np.arange(50, 91, 2)):
    d = df.copy()
    d = d[np.isfinite(d['age']) & np.isfinite(d[y_col])]
    d['age_bin'] = pd.cut(d['age'], bins=bin_edges, include_lowest=True, right=False)
    agg = d.groupby(['diagnosis', 'age_bin'], observed=False)[y_col].agg(['mean', 'std', 'count']).reset_index()
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    mp = {cat: c for cat, c in zip(d['age_bin'].cat.categories, centers)}
    agg['age_center'] = agg['age_bin'].map(mp).astype(float)
    return agg

pred_obs_fc = PRED_DF[PRED_DF['phase'].isin(['observed', 'forecast'])].copy()
agg_th = age_binned_curve(pred_obs_fc, 'thickness_pred_proxy')
agg_bu = age_binned_curve(pred_obs_fc, 'bump_pred_proxy')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for diag, color, label in [(0, 'tab:blue', 'Healthy'), (1, 'tab:red', 'Diseased')]:
    a = agg_th[agg_th['diagnosis'] == diag].sort_values('age_center')
    x = a['age_center'].to_numpy()
    y = a['mean'].to_numpy()
    s = a['std'].fillna(0.0).to_numpy()
    axes[0].plot(x, y, marker='o', color=color, linewidth=2.2, label=label)
    axes[0].fill_between(x, y - 0.5 * s, y + 0.5 * s, color=color, alpha=0.18)

for diag, color, label in [(0, 'tab:blue', 'Healthy'), (1, 'tab:red', 'Diseased')]:
    a = agg_bu[agg_bu['diagnosis'] == diag].sort_values('age_center')
    x = a['age_center'].to_numpy()
    y = a['mean'].to_numpy()
    s = a['std'].fillna(0.0).to_numpy()
    axes[1].plot(x, y, marker='o', color=color, linewidth=2.2, label=label)
    axes[1].fill_between(x, y - 0.5 * s, y + 0.5 * s, color=color, alpha=0.18)

for diag, color, label in [(0, 'tab:blue', 'Healthy'), (1, 'tab:red', 'Diseased')]:
    d = PRED_DF[PRED_DF['diagnosis'] == diag]
    axes[2].scatter(d['age'], d['latent_speed'], s=16, alpha=0.30, color=color, label=label)

axes[0].set_title('Predicted thickness proxy vs age')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('thickness proxy')
axes[0].legend()

axes[1].set_title('Predicted bump proxy vs age')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('bump proxy')
axes[1].legend()

axes[2].set_title('Predicted latent speed vs age')
axes[2].set_xlabel('Age')
axes[2].set_ylabel('||v||')
axes[2].legend()

plt.tight_layout()
plt.show()

# Observed vs forecast/future phase view (as in velocity results notebook style).
mask_obs = PRED_DF['phase'] == 'observed'
mask_fc = PRED_DF['phase'] == 'forecast'
mask_fu = PRED_DF['phase'] == 'future'

plt.figure(figsize=(10, 5))
plt.scatter(PRED_DF.loc[mask_obs, 'age'], PRED_DF.loc[mask_obs, 'latent_speed'], s=26, alpha=0.65, label='Observed', color='tab:purple')
plt.scatter(PRED_DF.loc[mask_fc, 'age'], PRED_DF.loc[mask_fc, 'latent_speed'], s=26, alpha=0.60, label='Forecast', color='tab:red')
if mask_fu.any():
    plt.scatter(PRED_DF.loc[mask_fu, 'age'], PRED_DF.loc[mask_fu, 'latent_speed'], s=28, alpha=0.60, label='Future extrapolated', color='tab:green')
plt.xlabel('Age')
plt.ylabel('Latent speed ||v||')
plt.title('Test latent speeds by phase')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

# Compare against label-space expected trends for thickness/bump.
if {'thickness_gt', 'bump_gt', 'age'}.issubset(PRED_DF.columns):
    has_gt = PRED_DF[np.isfinite(PRED_DF['thickness_gt']) & np.isfinite(PRED_DF['bump_gt'])]
    if len(has_gt) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].scatter(has_gt['thickness_gt'], has_gt['thickness_pred_proxy'], c=has_gt['diagnosis'], cmap='coolwarm', s=20, alpha=0.55)
        axes[0].set_xlabel('GT thickness_t')
        axes[0].set_ylabel('Pred thickness proxy')
        axes[0].set_title('Thickness: GT vs prediction proxy')

        axes[1].scatter(has_gt['bump_gt'], has_gt['bump_pred_proxy'], c=has_gt['diagnosis'], cmap='coolwarm', s=20, alpha=0.55)
        axes[1].set_xlabel('GT bump_height')
        axes[1].set_ylabel('Pred bump proxy')
        axes[1].set_title('Bump: GT vs prediction proxy')
        plt.tight_layout()
        plt.show()


In [ ]:

# ---- Export analysis tables ----

OUT_DIR = EXP_DIR / 'analysis_age_disease_forecast'
OUT_DIR.mkdir(parents=True, exist_ok=True)

pred_csv = OUT_DIR / 'test_forecast_metrics.csv'
summary_json = OUT_DIR / 'summary.json'

PRED_DF.to_csv(pred_csv, index=False)

summary = {
    'experiment_dir': str(EXP_DIR),
    'checkpoint': CHECKPOINT,
    'num_rows': int(len(PRED_DF)),
    'num_subjects': int(PRED_DF['subject_id'].nunique()),
    'diagnosis_counts': {str(k): int(v) for k, v in PRED_DF['diagnosis'].value_counts(dropna=False).to_dict().items()},
    'phase_counts': {str(k): int(v) for k, v in PRED_DF['phase'].value_counts(dropna=False).to_dict().items()},
    'time_key': TIME_KEY,
    'time_mode': TIME_MODE,
    'age_conditioning_keys': AGE_CONDITION_KEYS,
    'future_years': FUTURE_YEARS,
}
summary_json.write_text(json.dumps(summary, indent=2))

print('Saved:', pred_csv)
print('Saved:', summary_json)
print(json.dumps(summary, indent=2))
